In [1]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage
from typing import Annotated
from typing_extensions import TypedDict
import operator

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

model = ChatOpenAI(api_key="api_key", model="gpt-3.5-turbo")

# define a high risk task---- send email
def send_email(to: str, content: str) -> str:
    return f"email send to {to}, content:{content}"

tool_map = {"send_email": send_email}

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "email to a person",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "receiver"},
                    "content": {"type": "string", "description": "email content"}
                },
                "required": ["to", "content"]
            }
        }
    }
]

model_with_tools = model.bind_tools(tools_schema)

def llm_node(state: AgentState):
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_node(state: AgentState):
    last_message = state["messages"][-1]
    tool_results = []
    for tool_call in last_message.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]
        result = tool_map[name](**args)
        tool_results.append(
            ToolMessage(content=result, tool_call_id=tool_call["id"])
        )
    return {"messages": tool_results}

def should_continue(state: AgentState):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "call_tool"
    return "end"

graph_builder = StateGraph(AgentState)
graph_builder.add_node("llm", llm_node)
graph_builder.add_node("tool", tool_node)
graph_builder.set_entry_point("llm")
graph_builder.add_conditional_edges(
    "llm", should_continue, {"call_tool": "tool", "end": END}
)
graph_builder.add_edge("tool", "llm")

checkpointer = MemorySaver()

# 1. the key process is interrupt the loop, interrupt_before
agent = graph_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["tool"]   # pause before tool call
)
config = {"configurable": {"thread_id": "message_001"}}

# 2. it is will be interrupted before tool call
result = agent.invoke(
    {"messages": [HumanMessage("sent email to Alice, content: hi")]},
    config=config
)

# 3. check the agent what to do
last_message = result["messages"][-1]
print("Agent want:")
for tool_call in last_message.tool_calls:
    print(f"  call tool: {tool_call['name']}")
    print(f"  args: {tool_call['args']}")

# 4. human verify
user_decision = input("Do you want to continue? (yes/no): ")

if user_decision == "yes":
    # 5 continue the loop
    final_res = agent.invoke(None, config=config)
    print(final_res["messages"][-1].content)
else:
    print("Agent stopped.")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: api_key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}